In [ ]:
import json

def load_json_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

data_all = load_json_file("/root/private_data/luog/AbAgKer/data/data_aug/json/AbAgI_9678.json")
no_kd = []
for i in data_all:
    if i["AbAgA"] == 0:
        no_kd.append(i)

with open("/root/private_data/luog/AbAgKer/data/data_aug/json/no_kd.json", 'w', encoding='utf-8') as f:
    json.dump(no_kd, f, ensure_ascii=False, indent=2)


In [2]:

import os
os.chdir("/root/private_data/luog/AbAgKer")

import torch
import json
import random
import yaml
import numpy as np
from omegaconf import OmegaConf
from tqdm import tqdm

from taming.models.AbAgKer_newLLM import AbAgKerTrainer
from main import DataModuleFromConfig, instantiate_from_config


DEVICE = torch.device("cuda:0")

def load_config(config_path, display=False):
  config = OmegaConf.load(config_path)
  if display:
    print(yaml.dump(OmegaConf.to_container(config)))
  return config

def load_model(config, ckpt_path=None):
    model = AbAgKerTrainer(**config.model.params)
    model.to(DEVICE)
    if ckpt_path is not None:
        model.init_from_ckpt(ckpt_path)
    return model.eval()

config_path = "/root/private_data/luog/AbAgKer/logs/2025-11-21T13-07-39_AbAgKer_fold3_1120/configs/2025-11-21T13-07-39-project.yaml"
ckpt_path = "/root/private_data/luog/AbAgKer/logs/2025-11-21T13-07-39_AbAgKer_fold3_1120/AbAgker/2025-11-21T13-07-39_AbAgKer_fold3_1120/checkpoints/epoch=49-step=9600.ckpt"

config = load_config(config_path)
model = load_model(config, ckpt_path)


Using common tokenizer.
Loading pre-trained tokenizer...
Loading pre-trained tokenizer...


/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of EsmModel were not initialized from the model checkpoint at /root/private_data/luog/AbAgKer/pretrain_LLM/esm2 and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/conda/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)  # noqa: B028


Restored from /root/private_data/luog/AbAgKer/logs/2025-11-21T13-07-39_AbAgKer_fold3_1120/AbAgker/2025-11-21T13-07-39_AbAgKer_fold3_1120/checkpoints/epoch=49-step=9600.ckpt


In [ ]:

config_datapath = "/root/private_data/luog/AbAgKer/data/data_aug/json/data.yaml"
config_data = load_config(config_datapath)
data = instantiate_from_config(config_data.data)
data.prepare_data()
data.setup()

val_dataloader = data.val_dataloader()
print(f"Val dataloader created with {len(val_dataloader)} batches")


model.eval()
all_predictions = {}

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(val_dataloader, desc="Predicting binding sites")):
        HLX, ex_HXL, label = model.get_input(batch)
        kd_pre, _, aux_dict = model.model(HLX,ex_HXL)

        for i in range(len(batch["pdb"])):
            all_predictions[batch["pdb"][i]] = kd_pre[i].tolist()

for i in tqdm(no_kd):
    i["AbAgA"] = all_predictions[i["pdb"]]

with open("/root/private_data/luog/AbAgKer/data/data_aug/json/no_kd_with_pred.json", 'w', encoding='utf-8') as f:
    json.dump(no_kd, f, ensure_ascii=False, indent=2)


Val dataloader created with 483 batches


100%|██████████| 7725/7725 [00:00<00:00, 610913.11it/s]


In [4]:
all_predictions

{'5KQV_TONWH': 8.772732734680176,
 '8R9Y_HDACA': 7.4260969161987305,
 '7X2O_SBKIP': 8.352444648742676,
 '3PNW_VRFEJ': 9.870382308959961,
 '8ZR4_GLULW': 7.311956405639648,
 '8YWR_QIEHD': 10.588237762451172,
 '6MTN_RAURJ': 6.514944076538086,
 '6K6B_YLIWR': 8.363356590270996,
 '7COE_WMLPA': 7.428117752075195,
 '9K3P_PDZUJ': 8.882970809936523,
 '6W4S_FNYPE': 9.033123016357422,
 '6KO5_TFVNW': 9.663710594177246,
 '7BEL_YBJON': 7.544025421142578,
 '8Q5Y_ZUBJL': 7.6842265129089355,
 '7D5P_NWGCT': 9.116791725158691,
 '8YBO_HGVRH': 8.257440567016602,
 '8ZER_IFHIL': 8.247859001159668,
 '7TTM_RZIVF': 6.893102645874023,
 '6TNP_GPBZN': 8.410001754760742,
 '7T25_IEZRR': 9.71361255645752,
 '7COE_RJBBM': 6.755171298980713,
 '7QT3_PQKSL': 6.153185844421387,
 '4YGA_NWWFN': 9.013744354248047,
 '8SIS_YNURL': 8.124735832214355,
 '8CXG_USWGV': 7.43065881729126,
 '1JGU_AOOHE': 7.513218402862549,
 '9FVE_FRJDX': 8.50059700012207,
 '9ASD_FBTKL': 8.859285354614258,
 '7B27_DCCNR': 7.703152656555176,
 '7DKJ_FYJTA':

In [ ]:
import os
for root, dirs, files in os.walk("/root/private_data/luog/AbAgKer/data/data_aug/AbAgA_byPDB_train"):
    for file in files:
        if file.endswith(".json"):
            with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                data_origin = json.load(f)
            with open("/root/private_data/luog/AbAgKer/data/data_aug/json/no_kd_with_pred.json", 'r', encoding='utf-8') as f:
                no_kd = json.load(f)
            data = data_origin + no_kd
            with open(os.path.join("/root/private_data/luog/AbAgKer/data/data_aug/AbAgA_byPDB_train_addIdata", file), 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
                
                        


In [9]:
import json
a = json.load(open("/root/private_data/luog/AbAgKer/data/data_aug/AbAgA_byPDB_train_addIdata/fold_1_train.json", 'r', encoding='utf-8'))
a

[{'pdb': '3LEV_SPXNO',
  'AbAgI': 1,
  'mutation': 'SabDab_noinfo',
  'AbAgA': 9.221848749616356,
  'AbAgAoff': 0.0,
  'X': 'SDPVRQYLHEIGEVLELDKWAELGAAAKVEEGMEAIKKLSEATGLDQELIREVVRAKILGTAAIQKIPGLKEKPDPKTVEEVDGKLKSLPKELKRYLHIAREGEAARQHLIEANLRLVVSIAKKYTGRGLSFLDLIQEGNQGLIRAVEKFEYKRGFAFSTYATWWIRQAINRAIADQ',
  'H': 'RITLKESGPPLVKPTQTLTLTCSFSGFSLSDFGVGVGWIRQPPGKALEWLAIIYSDDDKRYSPSLNTRLTITKDTSKNQVVLVMTRVSPVDTATYFCAHRRGPTTLFGVPIARGPVNAMDVWGQGITVTISSTSTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDK',
  'L': 'ALQLTQSPSSLSASVGDRITITCRASQGVTSALAWYRQKPGSPPQLLIYDASSLESGVPSRFSGSGSGTEFTLTISTLRPEDFATYYCQQLHFYPHTFGGGTRVDVRRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC'},
 {'pdb': '6H7N_TLWYE',
  'AbAgI': 1,
  'mutation': 'SabDab_noinfo',
  'AbAgA': 8.455931955649724,
  'AbAgAoff': 0.0,
  'X': 'AAKVMSLLMALVVLLIVAGNVLVIAAIGSTQRLQTLTNLFITSLACADLVVGLLVVPFGATLVVRGTWLWGSFLCELWTSLDVLCVTASIETLCVI